# 03 - Cleaning Verification & Sanity Audit

This notebook audits the output of `src/clean.py` against the raw interim extraction (`2024-01`).
It validates:
1. Dataset integrity, row attrition, and duplicate elimination
2. Chronological assertions ($t_{req} \le t_{pickup} < t_{dropoff}$)
3. Operational and telematics boundaries (duration, distance, implied speed)
4. Economic sanity (positive base fare and driver pay)
5. Volume distributions across all target venue and spillover zones

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

# Path resolution for interim raw vs cleaned
INTERIM_DIR = Path('../data/interim') if Path('../data/interim').exists() else Path('data/interim')

raw_path = INTERIM_DIR / 'venue_trips_2024-01.parquet'
clean_path = INTERIM_DIR / 'cleaned_venue_trips_2024-01.parquet'

assert raw_path.exists(), f'Missing raw interim file: {raw_path}'
assert clean_path.exists(), f'Missing cleaned file: {clean_path}'

df_raw = pd.read_parquet(raw_path)
df_clean = pd.read_parquet(clean_path)

print(f'Raw Interim Records:   {len(df_raw):,}')
print(f'Cleaned Records:       {len(df_clean):,}')
print(f'Records Filtered Out:  {len(df_raw) - len(df_clean):,} ({(len(df_raw) - len(df_clean)) / len(df_raw) * 100:.2f}%)')
print(f'Overall Retention:     {len(df_clean) / len(df_raw) * 100:.2f}%')

### 1. Deduplication and Null Audits

In [ ]:
raw_dupes = df_raw.duplicated().sum()
clean_dupes = df_clean.duplicated().sum()
clean_nulls = df_clean.isna().sum().sum()

print(f'Raw exact duplicates:      {raw_dupes:,}')
print(f'Clean exact duplicates:    {clean_dupes:,} (Expected: 0)')
print(f'Clean total null values:   {clean_nulls:,} (Expected: 0)')

assert clean_dupes == 0, 'Cleaned dataset contains unexpected duplicates!'
assert clean_nulls == 0, 'Cleaned dataset contains unexpected nulls!'

### 2. Chronological Integrity
Verifies: $\text{request\_datetime} \le \text{pickup\_datetime} < \text{dropoff\_datetime}$

In [ ]:
req_after_pickup = (df_clean['request_datetime'] > df_clean['pickup_datetime']).sum()
pickup_after_dropoff = (df_clean['pickup_datetime'] >= df_clean['dropoff_datetime']).sum()

print(f'Violations (request > pickup):     {req_after_pickup}')
print(f'Violations (pickup >= dropoff):    {pickup_after_dropoff}')

assert req_after_pickup == 0, 'Found trips requested after pickup!'
assert pickup_after_dropoff == 0, 'Found trips picked up after or at dropoff!'

### 3. Operational & Telematics Bounds
Checking threshold compliance:
- $60\text{ s} \le \text{trip\_time} \le 14,400\text{ s}$
- $0.1\text{ mi} \le \text{trip\_miles} \le 100.0\text{ mi}$
- $\text{implied\_speed} \le 75.0\text{ mph}$

In [ ]:
implied_speed = df_clean['trip_miles'] / (df_clean['trip_time'] / 3600.0)

bounds_audit = pd.DataFrame({
    'Metric': ['trip_time (sec)', 'trip_miles', 'implied_speed (mph)'],
    'Min': [df_clean['trip_time'].min(), df_clean['trip_miles'].min(), implied_speed.min()],
    'Median': [df_clean['trip_time'].median(), df_clean['trip_miles'].median(), implied_speed.median()],
    'Max': [df_clean['trip_time'].max(), df_clean['trip_miles'].max(), implied_speed.max()],
    'Allowed Min': [60, 0.1, 0.0],
    'Allowed Max': [14400, 100.0, 75.0]
})
bounds_audit.round(2)

### 4. Economic Bounds
Verifies: $\text{base\_passenger\_fare} \ge 0.01$ and $\text{driver\_pay} \ge 0.00$

In [ ]:
invalid_fares = (df_clean['base_passenger_fare'] <= 0).sum()
invalid_pay = (df_clean['driver_pay'] < 0).sum()

print(f'Trips with non-positive base fare: {invalid_fares}')
print(f'Trips with negative driver pay:     {invalid_pay}')

assert invalid_fares == 0, 'Found non-positive fares!'
assert invalid_pay == 0, 'Found negative driver pay!'

### 5. Zone Volume Attrition Comparison
Verifies that cleaning did not disproportionately drop records from any single venue or spillover zone.

In [ ]:
zone_labels = {
    189: 'Barclays Core (189)',
    49:  'Barclays Spill (49)',
    62:  'Barclays Spill (62)',
    69:  'Barclays Spill (69)',
    97:  'Barclays Spill (97)',
    181: 'Barclays Spill (181)',
    190: 'Barclays Spill (190)',
    253: 'Citi Field Core (253)',
    93:  'Citi Field Spill (93)'
}

raw_counts = df_raw['PULocationID'].value_counts()
clean_counts = df_clean['PULocationID'].value_counts()

comparison = pd.DataFrame({
    'Zone Name': pd.Series(zone_labels),
    'Raw Trips': raw_counts,
    'Clean Trips': clean_counts
}).fillna(0)

comparison['Dropped'] = comparison['Raw Trips'] - comparison['Clean Trips']
comparison['Retention (%)'] = (comparison['Clean Trips'] / comparison['Raw Trips'] * 100).round(2)
comparison.sort_index()